# Notebook 21 — Kaggle Battle Agent

## The Pokémon Company — PTCG AI Battle Challenge

### Team Jesus

This notebook wraps the completed policy engine in a production-facing agent
that can be called by the official Kaggle simulator.

## Existing pipeline

```text
Official observation dictionary
        ↓
to_observation_class()
        ↓
Notebook 18 observation adapter
        ↓
BattleSnapshot
        ↓
Notebook 19 feature and scoring engine
        ↓
Notebook 20 BattlePolicy
        ↓
Best official option index

## Notebook 21 objectives
Load the official cg runtime.
Load the completed Notebook 20 policy engine.
Build a Kaggle-compatible agent wrapper.
Handle initial deck requests.
Handle normal legal-action requests.
Return lists of official option indices.
Enforce minCount and maxCount.
Prevent duplicate option selections.
Add deterministic fallback behavior.
Track timing, decisions, errors, and fallback usage.
Expose a production agent(obs_dict) entry point.
Export reusable code into src/kaggle_agent/.

## Target structure

src/kaggle_agent/
├── __init__.py
├── models.py
├── runtime.py
├── agent.py
├── validation.py
└── notebook21_export.py

## Cell 2 — Imports

In [1]:
from __future__ import annotations

import sys
import time
import types

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Sequence

print("Python:", sys.version)
print("Current directory:", Path.cwd())

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks


## Cell 3 — Locate project paths

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    markers = [
        "src",
        "notebooks",
        "scripts",
        "data",
    ]

    for candidate in [current, *current.parents]:
        marker_count = sum(
            (candidate / marker).exists()
            for marker in markers
        )

        if marker_count >= 3:
            return candidate

    if current.name.lower() == "notebooks":
        return current.parent

    return current


PROJECT_ROOT = find_project_root()

SRC_DIR = PROJECT_ROOT / "src"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

NOTEBOOK20_EXPORT = (
    SCRIPTS_DIR
    / "20_policy_engine.py"
)

KAGGLE_AGENT_DIR = (
    SRC_DIR
    / "kaggle_agent"
)

NOTEBOOK21_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "notebook21"
)

for directory in [
    KAGGLE_AGENT_DIR,
    NOTEBOOK21_REPORT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Notebook 20 export:", NOTEBOOK20_EXPORT)
print("Kaggle agent package:", KAGGLE_AGENT_DIR)
print("Notebook 21 reports:", NOTEBOOK21_REPORT_DIR)

Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 20 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\20_policy_engine.py
Kaggle agent package: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\kaggle_agent
Notebook 21 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook21


## Cell 4 — Load Notebook 20 safely
### Notebook 20 is an exported notebook script, use the same safe loading method that worked previously

In [3]:
# Cell 4 — Load Notebook 20 and safely register nested notebook modules

import sys
import types
import uuid


if not NOTEBOOK20_EXPORT.is_file():
    raise FileNotFoundError(
        f"Notebook 20 export not found:\n{NOTEBOOK20_EXPORT}"
    )

source_20 = NOTEBOOK20_EXPORT.read_text(
    encoding="utf-8-sig"
)

source_20_lines = source_20.splitlines()

# Normalize repeated future imports.
cleaned_20_lines = [
    line
    for line in source_20_lines
    if line.strip() != "from __future__ import annotations"
]

cleaned_20_source = (
    "from __future__ import annotations\n"
    + "\n".join(cleaned_20_lines)
)

# Patch Notebook 20's nested importlib loaders.
# Dataclasses in Python 3.13 require the module to be registered
# in sys.modules before the module code executes.
cleaned_20_source = cleaned_20_source.replace(
    "notebook18 = importlib.util.module_from_spec(spec18)\n"
    "spec18.loader.exec_module(notebook18)",
    "notebook18 = importlib.util.module_from_spec(spec18)\n"
    "sys.modules[spec18.name] = notebook18\n"
    "spec18.loader.exec_module(notebook18)",
)

cleaned_20_source = cleaned_20_source.replace(
    "notebook19 = importlib.util.module_from_spec(spec19)\n"
    "spec19.loader.exec_module(notebook19)",
    "notebook19 = importlib.util.module_from_spec(spec19)\n"
    "sys.modules[spec19.name] = notebook19\n"
    "spec19.loader.exec_module(notebook19)",
)

module_name_20 = (
    "notebook20_policy_engine_"
    + uuid.uuid4().hex
)

notebook20 = types.ModuleType(module_name_20)
notebook20.__file__ = str(NOTEBOOK20_EXPORT)
notebook20.__package__ = ""

# Register Notebook 20 itself before execution.
sys.modules[module_name_20] = notebook20

compiled_20 = compile(
    cleaned_20_source,
    str(NOTEBOOK20_EXPORT),
    "exec",
)

exec(
    compiled_20,
    notebook20.__dict__,
)

print("\nNotebook 20 loaded from:")
print(notebook20.__file__)
print()
print(
    "PolicyDecision available:",
    hasattr(notebook20, "PolicyDecision"),
)
print(
    "BattlePolicy available:",
    hasattr(notebook20, "BattlePolicy"),
)
print(
    "Repository available:",
    hasattr(notebook20, "repository"),
)
print(
    "Official lookup available:",
    hasattr(
        notebook20,
        "official_card_data_by_id",
    ),
)

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 18 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\18_kaggle_observation_adapter.py
Notebook 19 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\19_battle_feature_extraction.py
Policy engine: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\policy_engine
Notebook 20 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook20
Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Card database: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Batt

,name,module,annotations
18,ApiResult,cg.api,"[state, error]"
1,AreaType,cg.api,[]
21,Attack,cg.api,"[attackId, name, text, damage, energies]"
9,Card,cg.api,"[id, serial, playerIndex]"
20,CardData,cg.api,"[cardId, name, cardType, retreatCost, hp, weak..."
3,CardType,cg.api,[]
2,EnergyType,cg.api,[]
0,IntEnum,enum,[]
15,Log,cg.api,"[type, playerIndex, hasBasicPokemon, cardId, s..."
8,LogType,cg.api,[]


Important official classes:

- Card
- CardData
- CardType
- Log
- LogType
- Option
- OptionType
- PlayerState
- Pokemon
- SearchState
- SelectContext
- SelectData
- SelectType
- State

Card
Class: <class 'cg.api.Card'>

Annotations:
- id: <class 'int'>
- serial: <class 'int'>
- playerIndex: <class 'int'>

CardData
Class: <class 'cg.api.CardData'>

Annotations:
- cardId: <class 'int'>
- name: <class 'str'>
- cardType: <enum 'CardType'>
- retreatCost: <class 'int'>
- hp: <class 'int'>
- weakness: cg.api.EnergyType | None
- resistance: cg.api.EnergyType | None
- energyType: <enum 'EnergyType'>
- basic: <class 'bool'>
- stage1: <class 'bool'>
- stage2: <class 'bool'>
- ex: <class 'bool'>
- megaEx: <class 'bool'>
- tera: <class 'bool'>
- aceSpec: <class 'bool'>
- evolvesFrom: str | None
- skills: list[cg.api.Skill]
- attacks: list[int]

CardType
Class: <enum 'CardType'>

No annotations found.

Public attributes:
- BASIC_ENERGY
- ITEM
- POKEMON
- SPECIAL_ENERGY
- STADIUM
- SUPPORTER
- TOOL
-

,name,value
0,DECK,1
1,HAND,2
2,DISCARD,3
3,ACTIVE,4
4,BENCH,5
5,PRIZE,6
6,STADIUM,7
7,ENERGY,8
8,TOOL,9
9,PRE_EVOLUTION,10


OptionType


...

SelectContext


,name,value
0,MAIN,0
1,SETUP_ACTIVE_POKEMON,1
2,SETUP_BENCH_POKEMON,2
3,SWITCH,3
4,TO_ACTIVE,4
5,TO_BENCH,5
6,TO_FIELD,6
7,TO_HAND,7
8,DISCARD,8
9,TO_DECK,9


Official cg card objects: 1267
Notebook 17 repository: 1267

Official unique Card IDs: 1267
Repository unique Card IDs: 1267
Missing from repository: 0
Extra in repository: 0
Official card sample 1
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 1
cardType: 5
energyType: 1
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {G} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 2
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 2
cardType: 5
energyType: 2
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {R} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 3
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 3
cardType: 5
energyType: 3
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {W} Energy
resistance: None
retr

,rank,option_index,option_type,semantic_label,score,reasons
0,1,0,ATTACK,Attack with Mega Lucario ex,259.913,base=100.0 | attack_bonus=150.0 | active_energ...
1,2,1,END,End Turn,-25.000,base=0.0 | end_turn_penalty=-25.0


[OK] player_feature_extraction
[OK] battle_feature_extraction
[OK] action_feature_extraction
[OK] legal_action_ranking
[OK] best_option_index

Notebook 19 validation passed.
Notebook 19 loaded.
Production functions imported.
True
True
True
True
True

Notebook dependencies verified.
Repository size: 1267
Official CardData lookup size: 1267

Policy dependencies loaded.
BattlePolicy created.
Debug mode: True
BattlePolicy created successfully.
Safe BattlePolicy created successfully.
Option index: 0
Fallback used: True
Reason: Observation adaptation failed: AttributeError: 'NoneType' object has no attribute 'current'

Fallback behavior passed.
Synthetic snapshot loaded.
Turn: 3
Legal options: 2

Synthetic snapshot validation passed.
BattlePolicy Decision
Chosen option: 0
Fallback used: False
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex

Ranked actions:
 1. index=0   type=ATTACK     score=  259.913 Attack with Mega Lucario ex
 2. index=1   type=END        score=

## Cell 5 — Retrieve Notebook 20 objects

In [4]:
required_names = [
    "PolicyDecision",
    "BattlePolicy",
    "repository",
    "official_card_data_by_id",
]

missing_names = [
    name
    for name in required_names
    if not hasattr(notebook20, name)
]

if missing_names:
    raise AttributeError(
        "Notebook 20 is missing:\n"
        + "\n".join(missing_names)
    )

PolicyDecision = notebook20.PolicyDecision
BattlePolicy = notebook20.BattlePolicy
repository = notebook20.repository

official_card_data_by_id = (
    notebook20.official_card_data_by_id
)

print("PolicyDecision:", PolicyDecision)
print("BattlePolicy:", BattlePolicy)
print("Repository size:", len(repository))
print(
    "Official lookup size:",
    len(official_card_data_by_id),
)

assert len(repository) == 1267
assert len(official_card_data_by_id) == 1267

print("\nNotebook 20 production objects loaded successfully.")

PolicyDecision: <class 'notebook20_policy_engine_9dd427c7685642009fcad518ea7067ae.PolicyDecision'>
BattlePolicy: <class 'notebook20_policy_engine_9dd427c7685642009fcad518ea7067ae.BattlePolicy'>
Repository size: 1267
Official lookup size: 1267

Notebook 20 production objects loaded successfully.


## Cell 6 — Verify the final policy class

In [5]:
required_policy_methods = [
    "decide",
    "decide_snapshot",
    "choose_action",
    "choose_snapshot_action",
    "print_decision",
]

missing_methods = [
    name
    for name in required_policy_methods
    if not hasattr(BattlePolicy, name)
]

for name in required_policy_methods:
    status = (
        "[OK]"
        if hasattr(BattlePolicy, name)
        else "[MISSING]"
    )
    print(status, name)

if missing_methods:
    raise AttributeError(
        "BattlePolicy is missing methods:\n"
        + "\n".join(missing_methods)
    )

print("\nFinal BattlePolicy version verified.")

[OK] decide
[OK] decide_snapshot
[OK] choose_action
[OK] choose_snapshot_action
[OK] print_decision

Final BattlePolicy version verified.


## Cell 7 — Instantiate the Notebook 21 policy

In [6]:
policy = BattlePolicy(
    repository=repository,
    official_lookup=official_card_data_by_id,
    debug=False,
    fallback_index=0,
)

assert callable(policy.decide)
assert callable(policy.choose_action)

print("Notebook 21 policy instance created successfully.")

Notebook 21 policy instance created successfully.


## Cell 8 — Create the Kaggle Agent wrapper

In [7]:
from typing import Any


def agent(observation: Any) -> int:
    """
    Kaggle entry point.

    Receives an official Kaggle Observation object and
    returns the selected legal action index.
    """

    return policy.choose_action(observation)


print("Kaggle agent created.")

Kaggle agent created.


## Cell 9 — Verify the wrapper

In [8]:
assert callable(agent)

print("Agent wrapper verified.")

Agent wrapper verified.


## Cell 10 — Test using our synthetic snapshot

### Unlike Notebook 20, Notebook 21 uses the production wrapper.

In [9]:
# Cell 10 — Safely retrieve and test the synthetic snapshot

if not hasattr(notebook20, "sample_snapshot"):
    raise AttributeError(
        "Notebook 20 does not expose sample_snapshot."
    )

sample_snapshot = notebook20.sample_snapshot

if sample_snapshot.selection is None:
    raise ValueError(
        "Synthetic snapshot has no selection data."
    )

best = policy.choose_snapshot_action(
    sample_snapshot
)

print("Turn:", sample_snapshot.turn)
print(
    "Legal options:",
    len(sample_snapshot.selection.options),
)
print("Best option:", best)

assert best == 0

print("\nSnapshot inference passed.")

Turn: 3
Legal options: 2
Best option: 0

Snapshot inference passed.


## Cell 11 — Build a production agent result model

In [10]:
from dataclasses import dataclass


@dataclass(frozen=True)
class AgentStats:
    calls: int = 0
    errors: int = 0
    fallbacks: int = 0
    total_seconds: float = 0.0

    @property
    def average_seconds(self) -> float:
        if self.calls == 0:
            return 0.0

        return self.total_seconds / self.calls

## Cell 12 — Build the KaggleBattleAgent wrapper

In [11]:
@dataclass
class KaggleBattleAgent:
    policy: BattlePolicy
    deck: tuple[int, ...]
    debug: bool = False

    calls: int = 0
    errors: int = 0
    fallbacks: int = 0
    total_seconds: float = 0.0

    def choose(self, observation: Any) -> list[int]:
        """
        Return a simulator-compatible list.

        Supports:
        1. An adapted BattleSnapshot for testing.
        2. An official initial deck request.
        3. A normal official Observation.
        """

        started = time.perf_counter()
        self.calls += 1

        try:
            # Adapted BattleSnapshot test path.
            if hasattr(observation, "selection"):
                decision = self.policy.decide_snapshot(
                    observation
                )

            # Official initial deck request.
            elif getattr(observation, "select", None) is None:
                return list(self.deck)

            # Normal official Observation.
            else:
                decision = self.policy.decide(
                    observation
                )

            if decision.used_fallback:
                self.fallbacks += 1

            result = [
                int(decision.option_index)
            ]

            if self.debug:
                print("Agent result:", result)
                print("Reason:", decision.reason)

            return result

        except Exception as exc:
            self.errors += 1
            self.fallbacks += 1

            if self.debug:
                print(
                    "Agent error:",
                    f"{type(exc).__name__}: {exc}",
                )

            return [0]

        finally:
            self.total_seconds += (
                time.perf_counter() - started
            )

    def stats(self) -> AgentStats:
        return AgentStats(
            calls=self.calls,
            errors=self.errors,
            fallbacks=self.fallbacks,
            total_seconds=self.total_seconds,
        )

## Cell 13 — Load the 60-card deck

In [12]:
from pathlib import Path


DECK_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "kaggle_sample_submission"
    / "deck.csv"
)

if not DECK_FILE.is_file():
    DECK_FILE = (
        PROJECT_ROOT
        / "submission_work"
        / "team_jesus_baseline"
        / "deck.csv"
    )

deck_ids = tuple(
    int(line.strip())
    for line in DECK_FILE.read_text(
        encoding="utf-8-sig"
    ).splitlines()
    if line.strip()
)

print("Deck file:", DECK_FILE)
print("Deck size:", len(deck_ids))

assert len(deck_ids) == 60

print("Deck loaded successfully.")

Deck file: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\data\raw\kaggle_sample_submission\deck.csv
Deck size: 60
Deck loaded successfully.


## Cell 14 — Instantiate the production agent

In [13]:
kaggle_agent = KaggleBattleAgent(
    policy=policy,
    deck=deck_ids,
    debug=True,
)

print("KaggleBattleAgent created.")
print("Deck size:", len(kaggle_agent.deck))

KaggleBattleAgent created.
Deck size: 60


## Cell 15 — Test deck submission mode

In [14]:
class DeckRequest:
    select = None

deck_request = DeckRequest()

deck_response = kaggle_agent.choose(deck_request)

print("Returned deck size:", len(deck_response))

assert len(deck_response) == 60
assert deck_response == list(deck_ids)

print("Deck submission passed.")

Returned deck size: 60
Deck submission passed.


## Cell 16 — Test action-selection mode

### Now verify that the production wrapper chooses the same action as the policy.

In [15]:
chosen = kaggle_agent.choose(sample_snapshot)

print("Returned action:", chosen)

assert chosen == [0]

print("Action selection passed.")

Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Returned action: [0]
Action selection passed.


## Cell 17 — Check runtime statistics

In [16]:
stats = kaggle_agent.stats()

print(stats)
print("Calls:", stats.calls)
print("Errors:", stats.errors)
print("Fallbacks:", stats.fallbacks)
print("Total seconds:", stats.total_seconds)
print("Average seconds:", stats.average_seconds)

assert stats.calls >= 2
assert stats.errors == 0
assert stats.fallbacks == 0
assert stats.average_seconds >= 0.0

print("\nAgent statistics passed.")

AgentStats(calls=2, errors=0, fallbacks=0, total_seconds=0.00022170000011101365)
Calls: 2
Errors: 0
Fallbacks: 0
Total seconds: 0.00022170000011101365
Average seconds: 0.00011085000005550683

Agent statistics passed.


## Cell 18 — Validate the production interface

In [17]:
notebook21_checks = {
    "policy_loaded": callable(policy.choose_action),
    "wrapper_callable": callable(agent),
    "production_agent_created": isinstance(
        kaggle_agent,
        KaggleBattleAgent,
    ),
    "deck_size": len(kaggle_agent.deck) == 60,
    "deck_submission": deck_response == list(deck_ids),
    "action_selection": chosen == [0],
    "no_runtime_errors": kaggle_agent.errors == 0,
}

for check, passed in notebook21_checks.items():
    print(
        f"{'[OK]' if passed else '[FAIL]'} "
        f"{check}"
    )

assert all(notebook21_checks.values())

print("\nNotebook 21 validation passed.")

[OK] policy_loaded
[OK] wrapper_callable
[OK] production_agent_created
[OK] deck_size
[OK] deck_submission
[OK] action_selection
[OK] no_runtime_errors

Notebook 21 validation passed.


## Cell 19 — Create the final callable entry point

#### The competition sample expects a list of indices, not a single integer.

In [18]:
def production_agent(observation: Any) -> list[int]:
    """
    Final Kaggle-facing agent entry point.
    """

    return kaggle_agent.choose(observation)


assert callable(production_agent)

print("Production agent entry point created.")

Production agent entry point created.


## Cell 20 — Final summary

In [19]:
print("=" * 72)
print("Notebook 21 — Kaggle Battle Agent")
print("=" * 72)
print("Repository cards:", len(repository))
print("Official cards:", len(official_card_data_by_id))
print("Deck size:", len(deck_ids))
print("Agent calls:", kaggle_agent.calls)
print("Agent errors:", kaggle_agent.errors)
print("Agent fallbacks:", kaggle_agent.fallbacks)
print("Deck path validated:", len(deck_response) == 60)
print("Action path validated:", chosen == [0])
print()
print("NOTEBOOK 21 COMPLETED SUCCESSFULLY")
print("Ready for PowerShell export and packaging.")

Notebook 21 — Kaggle Battle Agent
Repository cards: 1267
Official cards: 1267
Deck size: 60
Agent calls: 2
Agent errors: 0
Agent fallbacks: 0
Deck path validated: True
Action path validated: True

NOTEBOOK 21 COMPLETED SUCCESSFULLY
Ready for PowerShell export and packaging.
